In [1]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

In [2]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from flatten_json import flatten
from tqdm import tqdm
client = bigquery.Client.from_service_account_json('/home/analytics/.secure_files/hitwicketsuperstars-f3e8c620a88c.json')
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname'])
#warnings.filterwarnings('ignore')

In [151]:
# Time
start = dt.datetime(2019,6,8)
end = dt.datetime(2019,6,15)
print(start,end,end-start)

2019-06-08 00:00:00 2019-06-15 00:00:00 7 days, 0:00:00


# User collectibles

In [152]:
c_users = cursor.superstars.users
aw_users = []
for documents in c_users.find({'created_at': {'$lt': end, '$gte': start}},{"sign_up_details":1, "created_at":1,"login_details":1}): # end condition
    aw_users.append(documents)
dic_flattened = [flatten(d) for d in aw_users]
users = pd.DataFrame(dic_flattened)
users = users[["_id","created_at","sign_up_details_device_id","login_details_last_request_at"]]
users.columns = ["user_id","create_time","device_id","last_request"]
len(users)

769

In [153]:
users.sort_values(['device_id','create_time'],ascending=False,inplace=True)
users.drop_duplicates('device_id',inplace=True)
len(users)

700

In [154]:
users['d1'] =(users['last_request']-users['create_time']) > '24:00:00'
users.head()

,user_id,create_time,device_id,last_request,d1
610,5d0384bce9771100184fae7e,2019-06-14 11:27:56.873,ffec289af74ccae1ca25393407ac31cb,2019-06-14 11:28:44.459,False
670,5d03b56fad6fe100184ee962,2019-06-14 14:55:43.452,ffc3825df3835a737ecb9231fecdf5fa,2019-06-15 08:43:03.352,False
220,5cfd3544bba72a0018ced03d,2019-06-09 16:35:16.446,febe5ec86d338860145a25327fda34eb,2019-06-09 16:37:19.027,False
58,5cfb7d5dbba72a001844e17a,2019-06-08 09:18:21.392,fe8de6c0f660c25fc6b4a6c5a0cc9864,NaT,False
660,5d03a9b3b1b9f800115cf639,2019-06-14 14:05:39.353,fe37e2b988a1d16946319c474096db32,2019-06-14 14:38:58.939,False


In [155]:
#comment whatever kind of user is not needed

users = users[(users['last_request']-users['create_time'])>'00:05:00'] # only for those users who are atleast in the app for 5 mins after FTUE completion

#users = users[(users['last_request']-users['create_time'])>'24:00:00'] # only for those users who are atleast in the app for 24 hours after FTUE completion
print(len(users))

324


In [156]:
(923-409)/923*100

55.687973997833154

# team ID of each user id

In [157]:
team_cursor = cursor.superstars.teams
aw_team = []
for documents in team_cursor.find({'created_at': {'$lt': end, '$gte': start}},{"user":1,'created_at':1}): 
    aw_team.append(documents)
dic_flattened = [flatten(d) for d in aw_team]
team = pd.DataFrame(dic_flattened)
teams = team[team["user"].isin(users["user_id"])]
teams = teams[["_id","user",'created_at']]
teams.columns = ["team_id", "user_id","create_time"]

In [158]:
print(len(teams))
teams.head()

324


,team_id,user_id,create_time
0,5cfafb541c9cdd40cac99b2e,5cfafb541c9cdd40cac99b22,2019-06-08 00:03:32.670
1,5cfafd9e59e7ea40f010d33b,5cfafd9e59e7ea40f010d32f,2019-06-08 00:13:18.230
2,5cfafe9302e17c437b1d0b16,5cfafe9302e17c437b1d0b0a,2019-06-08 00:17:23.115
3,5cfb016ee18f9e45c45c3bf6,5cfb016ee18f9e45c45c3bea,2019-06-08 00:29:34.783
9,5cfb19e76d6da94c81527410,5cfb19e76d6da94c81527404,2019-06-08 02:13:59.175


In [159]:
teams = pd.merge(users[['user_id','d1']],teams,on='user_id')

In [160]:
print(len(teams))
teams.head()

324


,user_id,d1,team_id,create_time
0,5d03b56fad6fe100184ee962,False,5d03b56fad6fe100184ee96e,2019-06-14 14:55:43.459
1,5d03a9b3b1b9f800115cf639,False,5d03a9b3b1b9f800115cf645,2019-06-14 14:05:39.360
2,5cfe4eabbba72a0018006ecc,True,5cfe4eabbba72a0018006ed8,2019-06-10 12:35:55.242
3,5d03b31b5ea04600120cf52c,False,5d03b31b5ea04600120cf538,2019-06-14 14:45:47.944
4,5d0381d9b1b9f8001153e11c,False,5d0381d9b1b9f8001153e128,2019-06-14 11:15:37.623


In [161]:
player_cursor = cursor.superstars.players
aw_players = []
for documents in player_cursor.find({'created_at': {'$gte': start}},{'team':1}):
    aw_players.append(documents)
dic_flattened = [flatten(d) for d in aw_players]
players = pd.DataFrame(dic_flattened)
players = players[players["team"].isin(teams["team_id"])]
players = players[["_id","team"]]
players.columns = ["player_id", "team_id"]

In [162]:
print(len(players))
players.head()

1166


,player_id,team_id
0,5cfafb541c9cdd40cac99b32,5cfafb541c9cdd40cac99b2e
1,5cfafb541c9cdd40cac99b34,5cfafb541c9cdd40cac99b2e
2,5cfafcd259e7ea40f010cf72,5cfafb541c9cdd40cac99b2e
3,5cfafd9e59e7ea40f010d33f,5cfafd9e59e7ea40f010d33b
4,5cfafd9e59e7ea40f010d341,5cfafd9e59e7ea40f010d33b


In [163]:
users_team_player = pd.merge(teams,players,on='team_id')
print(len(users_team_player))
users_team_player.head()

1166


,user_id,d1,team_id,create_time,player_id
0,5d03b56fad6fe100184ee962,False,5d03b56fad6fe100184ee96e,2019-06-14 14:55:43.459,5d03b56fad6fe100184ee972
1,5d03b56fad6fe100184ee962,False,5d03b56fad6fe100184ee96e,2019-06-14 14:55:43.459,5d03b56fad6fe100184ee974
2,5d03b56fad6fe100184ee962,False,5d03b56fad6fe100184ee96e,2019-06-14 14:55:43.459,5d03b6c6de1a9600119e65d8
3,5d03b56fad6fe100184ee962,False,5d03b56fad6fe100184ee96e,2019-06-14 14:55:43.459,5d04adfea4732000125b7d7b
4,5d03a9b3b1b9f800115cf639,False,5d03a9b3b1b9f800115cf645,2019-06-14 14:05:39.360,5d03a9b3b1b9f800115cf649


In [164]:
skill_cursor = cursor.superstars.player_skill_logs
aw_skill = []
for documents in skill_cursor.aggregate([{'$unwind':"$details"}, 
                                    {"$match" : {'created_at': {'$gte': start}}},
                                     {"$match" : {"details.type" : 'TRAINING_PROGRESS'}}]):
    aw_skill.append(documents)
dic_flattened = [flatten(d) for d in aw_skill]
players_trained = pd.DataFrame(dic_flattened)
players_trained = players_trained[players_trained["player"].isin(players["player_id"])]
players_trained = players_trained[["created_at","details__id","player"]]
players_trained.columns = ["trained_at","training_id","player_id"]

In [165]:
print(len(players_trained))
players_trained.head()

3832


,trained_at,training_id,player_id
0,2019-06-08 00:23:21.275,5cfafff944b62545a9fe851b,5cfafb541c9cdd40cac99b32
1,2019-06-08 00:23:21.275,5cfb001644b62545a9fe8758,5cfafb541c9cdd40cac99b32
2,2019-06-08 00:23:21.275,5cfb0043a16d3245a3914af0,5cfafb541c9cdd40cac99b32
3,2019-06-08 00:23:21.275,5cfb006281c5d045cb07fe26,5cfafb541c9cdd40cac99b32
4,2019-06-08 00:23:21.275,5cfb0089a16d3245a3914ec3,5cfafb541c9cdd40cac99b32


In [166]:
# only for those who have trained
users_team_player_trained = pd.merge(users_team_player[['user_id','player_id','create_time','d1']],
                                    players_trained, on='player_id')
users_team_player_trained = users_team_player_trained[(users_team_player_trained['trained_at']-users_team_player_trained['create_time'])<'24:00:00']

In [167]:
print(len(users_team_player_trained))
users_team_player_trained.head()

3105


,user_id,player_id,create_time,d1,trained_at,training_id
0,5d03b56fad6fe100184ee962,5d03b56fad6fe100184ee972,2019-06-14 14:55:43.459,False,2019-06-15 08:37:39.583,5d04ae53d06f3100191a3b92
1,5d03b56fad6fe100184ee962,5d03b6c6de1a9600119e65d8,2019-06-14 14:55:43.459,False,2019-06-15 08:38:16.802,5d04ae78d06f3100191a4caf
2,5d03b56fad6fe100184ee962,5d04adfea4732000125b7d7b,2019-06-14 14:55:43.459,False,2019-06-15 08:37:13.907,5d04ae39a4732000125b8a78
3,5d03a9b3b1b9f800115cf639,5d03a9b3b1b9f800115cf649,2019-06-14 14:05:39.360,False,2019-06-14 14:08:02.233,5d03aa42e97711001856ddd9
4,5d03a9b3b1b9f800115cf639,5d03a9b3b1b9f800115cf649,2019-06-14 14:05:39.360,False,2019-06-14 14:08:02.233,5d03ad25e97711001857c5b9


In [168]:
total_trained = users_team_player_trained.groupby('user_id').agg({'training_id':'count','d1':'first'})
total_trained.columns = ['times_trained','d1']
total_trained.sort_values('times_trained',ascending=False,inplace=True)

In [169]:
total_trained = total_trained.reset_index()
print(len(total_trained))

total_trained.head()

217


,user_id,times_trained,d1
0,5cfbbe28bba72a001861ccbb,65,True
1,5d00524eaee56600118f48af,62,True
2,5d02c4587ed1b0001854429b,53,True
3,5d00e5bb1052b70011f50dd6,50,True
4,5cfd069209e8ab0011320433,47,True


In [170]:
total_trained.describe()

,times_trained
count,217.000000
mean,14.308756
std,12.797264
min,1.000000
25%,4.000000
50%,11.000000
75%,21.000000
max,65.000000


In [171]:
total_trained.head()

,user_id,times_trained,d1
0,5cfbbe28bba72a001861ccbb,65,True
1,5d00524eaee56600118f48af,62,True
2,5d02c4587ed1b0001854429b,53,True
3,5d00e5bb1052b70011f50dd6,50,True
4,5cfd069209e8ab0011320433,47,True


In [172]:
# run this to include the number of users who have trained 0 players

total_trained = pd.merge(total_trained[['user_id','times_trained']],users[['user_id','d1']],on='user_id',how='right')
total_trained = total_trained.fillna(0)

In [173]:
total_trained.tail()

,user_id,times_trained,d1
319,5d038372b1b9f8001154003c,0.0,False
320,5d03c634a00eaa001207227c,0.0,False
321,5cfb5a64bba72a0018340b9a,0.0,True
322,5d03d7c2afe99c00119f4857,0.0,False
323,5cffb865aee566001176461c,0.0,False


In [174]:
total_trained.sort_values('times_trained',ascending=False,inplace=True)
#total_trained['times_trained'] = np.where(total_trained['times_trained']>4,'5+', total_trained['times_trained'])
total_trained.tail()

,user_id,times_trained,d1
248,5d03caee0db57f0011839272,0.0,False
247,5cfcba6fbba72a0018ad44d9,0.0,True
246,5cfc07a4bba72a0018828e51,0.0,False
245,5d0251f21d1441001186f287,0.0,False
323,5cffb865aee566001176461c,0.0,False


In [175]:
number_of_training_before = total_trained.groupby('times_trained').agg({'user_id':'count','d1':'sum'})
number_of_training_before.columns = ['number_of_users','d1_users']

In [176]:
number_of_training_before['percentage_of_users'] = (number_of_training_before['number_of_users']/len(teams)*100).round()
number_of_training_before['percentage_of_d1_users'] = (number_of_training_before['d1_users']/number_of_training_before['number_of_users']*100)

In [177]:
print(len(number_of_training_before))
number_of_training_before

45


,number_of_users,d1_users,percentage_of_users,percentage_of_d1_users
times_trained,,,,
0.0,107,33.0,33.0,30.841121
1.0,16,1.0,5.0,6.250000
2.0,16,5.0,5.0,31.250000
3.0,17,1.0,5.0,5.882353
4.0,12,5.0,4.0,41.666667
5.0,13,1.0,4.0,7.692308
6.0,5,2.0,2.0,40.000000
7.0,9,3.0,3.0,33.333333
8.0,9,1.0,3.0,11.111111
